# 02 — Análise Exploratória de Dados (EDA)

**Objetivo:** caracterizar o dataset de licitações públicas federais de 2022, identificar padrões estruturais e produzir uma **lista preliminar de candidatos a anomalia** que servirá como referência semi-supervisionada para validação dos modelos.

**Estrutura do notebook:**
1. Setup e carregamento
2. Filtro de qualidade
3. Caracterização geral
4. Distribuição de valores
5. Modalidades de licitação
6. Análise temporal
7. Top fornecedores e concentração de mercado
8. Competição (número de participantes)
9. Lista preliminar de candidatos a anomalia

**Como usar este notebook:** os blocos `📝 **Análise:**` são placeholders para você escrever interpretações em prosa. Esse texto vai virar conteúdo direto do relatório final.

## 1. Setup e carregamento

Importa bibliotecas, configura estilo dos gráficos e carrega os 4 parquets tratados.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import INTERIM_DIR, CHAVE_LICITACAO

# Estilo dos gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

# Carregamento
df_lic = pd.read_parquet(INTERIM_DIR / "licitacao_tratado.parquet")
df_par = pd.read_parquet(INTERIM_DIR / "participanteslicitacao_tratado.parquet")
df_itens = pd.read_parquet(INTERIM_DIR / "itemlicitacao_tratado.parquet")
df_emp = pd.read_parquet(INTERIM_DIR / "empenhosrelacionados_tratado.parquet")

print(f"Licitações:    {len(df_lic):,} linhas")
print(f"Participantes: {len(df_par):,} linhas")
print(f"Itens:         {len(df_itens):,} linhas")
print(f"Empenhos:      {len(df_emp):,} linhas")
print(f"\nChave primária: {CHAVE_LICITACAO}")

ImportError: cannot import name 'CHAVE_LICITACAO' from 'src.config' (C:\Users\Gabriela\Pmonkey\IA\predicao\src\config.py)

## 2. Filtro de qualidade

**Decisão metodológica:** excluir licitações com valor zero, que correspondem a processos em fases intermediárias (publicação, suspensão, anulação). Não representam contratações efetivadas e contaminariam a detecção de anomalias de valor.

In [ ]:
n_antes = len(df_lic)
df_lic = df_lic[df_lic["valor_licitacao"] > 0].copy()
n_depois = len(df_lic)

print(f"Antes:     {n_antes:,} licitações")
print(f"Depois:    {n_depois:,} licitações")
print(f"Removidas: {n_antes - n_depois:,} ({(n_antes - n_depois)/n_antes*100:.2f}%)")

> 📝 **Análise (você escreve aqui):**
>
> Comente sobre o filtro aplicado, por que ele é metodologicamente correto, e qual é o dataset analítico resultante.

## 3. Caracterização geral

Estatísticas básicas para abrir o relatório: número de licitações, período coberto, abrangência geográfica, modalidades.

In [ ]:
print("=" * 60)
print("CARACTERIZAÇÃO DO DATASET")
print("=" * 60)
print(f"\nTotal de licitações efetivadas: {len(df_lic):,}")
print(f"Período: {df_lic['data_resultado_compra'].min().date()} a "
      f"{df_lic['data_resultado_compra'].max().date()}")
print(f"\nUFs presentes: {df_lic['uf'].nunique()}")
print(f"Órgãos distintos: {df_lic['nome_orgao'].nunique():,}")
print(f"UGs distintas: {df_lic['codigo_ug'].nunique():,}")
print(f"Modalidades: {df_lic['modalidade_compra'].nunique()}")

print(f"\nValor total movimentado: R$ {df_lic['valor_licitacao'].sum()/1e9:,.2f} bilhões")
print(f"Valor médio:  R$ {df_lic['valor_licitacao'].mean():,.2f}")
print(f"Valor mediano: R$ {df_lic['valor_licitacao'].median():,.2f}")

In [ ]:
# Distribuição por UF (top 15)
fig, ax = plt.subplots(figsize=(12, 5))
top_uf = df_lic["uf"].value_counts().head(15)
top_uf.plot.bar(ax=ax, color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_title("Top 15 UFs por número de licitações")
ax.set_ylabel("Número de licitações")
ax.set_xlabel("UF")
plt.tight_layout()
plt.show()

> 📝 **Análise:**
>
> Comente o volume total, a concentração geográfica e o porte do dataset. Há concentração esperada em DF (sede de muitos órgãos federais)?

## 4. Distribuição de valores

Valores de contratos públicos têm **cauda extremamente longa** (razão máximo/mediana de ~540.000x neste dataset). Análise linear é ilegível. A escala logarítmica revela a estrutura real.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escala linear
axes[0].hist(df_lic["valor_licitacao"], bins=80, color="coral", edgecolor="black", linewidth=0.3)
axes[0].set_title("Distribuição de valores — escala LINEAR")
axes[0].set_xlabel("Valor (R$)")
axes[0].set_ylabel("Frequência")
axes[0].ticklabel_format(style="plain", axis="x")

# Escala logarítmica
log_valores = np.log10(df_lic["valor_licitacao"])
axes[1].hist(log_valores, bins=80, color="teal", edgecolor="black", linewidth=0.3)
axes[1].set_title("Distribuição de valores — escala LOG10")
axes[1].set_xlabel("log10(Valor em R$)")
axes[1].set_ylabel("Frequência")

# Linhas de referência em valores notáveis
for valor, label in [(1e3, "R$ 1 mil"), (1e6, "R$ 1 mi"), (1e9, "R$ 1 bi")]:
    axes[1].axvline(np.log10(valor), color="red", linestyle="--", alpha=0.5)
    axes[1].text(np.log10(valor), axes[1].get_ylim()[1]*0.9, label,
                 rotation=90, ha="right", fontsize=9, color="red")

plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas descritivas detalhadas
print("Estatísticas de Valor Licitação (R$):")
print(f"  Mínimo:   {df_lic['valor_licitacao'].min():>20,.2f}")
print(f"  P10:      {df_lic['valor_licitacao'].quantile(0.10):>20,.2f}")
print(f"  P25:      {df_lic['valor_licitacao'].quantile(0.25):>20,.2f}")
print(f"  Mediana:  {df_lic['valor_licitacao'].median():>20,.2f}")
print(f"  Média:    {df_lic['valor_licitacao'].mean():>20,.2f}")
print(f"  P75:      {df_lic['valor_licitacao'].quantile(0.75):>20,.2f}")
print(f"  P90:      {df_lic['valor_licitacao'].quantile(0.90):>20,.2f}")
print(f"  P99:      {df_lic['valor_licitacao'].quantile(0.99):>20,.2f}")
print(f"  Máximo:   {df_lic['valor_licitacao'].max():>20,.2f}")

# Detecção do "limite mágico" da Lei 14.133/2021 (dispensa de licitação)
print(f"\nLicitações exatamente em R$ 17.600: {(df_lic['valor_licitacao'] == 17600).sum()}")
print(f"Licitações entre R$ 17.500 e R$ 17.700: "
      f"{((df_lic['valor_licitacao'] >= 17500) & (df_lic['valor_licitacao'] <= 17700)).sum()}")

> 📝 **Análise:**
>
> Discuta o que a escala logarítmica revela. Mencione a razão máximo/mediana de ~540.000x e o que isso implica para modelagem. Comente acúmulos suspeitos em valores específicos (R$ 17.600 é o limite de dispensa por valor da Lei 14.133/2021).

## 5. Modalidades de licitação

Modalidades diferentes têm regimes legais distintos. **Comparar valores entre modalidades exige cuidado** — uma dispensa de R$ 2 milhões e um pregão de R$ 2 milhões são juridicamente muito diferentes.

In [ ]:
# Composição
modalidades = df_lic["modalidade_compra"].value_counts()
print("Distribuição por modalidade:")
for mod, count in modalidades.items():
    pct = count/len(df_lic)*100
    print(f"  {mod:<40} {count:>7,}  ({pct:.2f}%)")

In [ ]:
# Boxplot de valores por modalidade (escala log)
df_lic_plot = df_lic.copy()
df_lic_plot["log_valor"] = np.log10(df_lic_plot["valor_licitacao"])

# Ordenar modalidades pela mediana do valor
ordem = (
    df_lic_plot.groupby("modalidade_compra")["log_valor"]
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_lic_plot, x="modalidade_compra", y="log_valor",
            order=ordem, ax=ax, palette="Set2", showfliers=True)
ax.set_title("Distribuição de valores por modalidade (escala log10)")
ax.set_xlabel("Modalidade")
ax.set_ylabel("log10(Valor em R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Tabela resumo: valor mediano e máximo por modalidade
resumo_modalidade = (
    df_lic.groupby("modalidade_compra")["valor_licitacao"]
    .agg(["count", "median", "mean", "max"])
    .sort_values("median", ascending=False)
    .round(2)
)
resumo_modalidade.columns = ["N licitações", "Mediana (R$)", "Média (R$)", "Máximo (R$)"]
print("Resumo por modalidade:")
print(resumo_modalidade.to_string())

> 📝 **Análise:**
>
> Compare modalidades com competição (Pregão, Concorrência) versus sem competição (Dispensa, Inexigibilidade). Comente as medianas e amplitudes. Algum boxplot mostra outliers acima do padrão legal da modalidade?

## 6. Análise temporal

Padrões sazonais em contratação pública são esperados (pico de dezembro pelo fim de exercício orçamentário). **Picos atípicos podem indicar gastos eleitoreiros, urgências mal-planejadas, ou esquemas.**

In [ ]:
# Série temporal mensal
df_lic["mes"] = df_lic["data_resultado_compra"].dt.to_period("M")

serie = df_lic.groupby("mes").agg(
    n_licitacoes=("valor_licitacao", "count"),
    valor_total=("valor_licitacao", "sum")
)
serie.index = serie.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(serie.index, serie["n_licitacoes"], marker="o", color="steelblue", linewidth=2)
axes[0].fill_between(serie.index, serie["n_licitacoes"], alpha=0.2, color="steelblue")
axes[0].set_title("Número de licitações homologadas por mês")
axes[0].set_ylabel("Quantidade")
axes[0].grid(True, alpha=0.3)

axes[1].plot(serie.index, serie["valor_total"]/1e9, marker="o", color="darkgreen", linewidth=2)
axes[1].fill_between(serie.index, serie["valor_total"]/1e9, alpha=0.2, color="darkgreen")
axes[1].set_title("Valor total contratado por mês")
axes[1].set_ylabel("Valor total (R$ bilhões)")
axes[1].set_xlabel("Mês")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nValores por mês:")
serie_print = serie.copy()
serie_print["valor_total"] = (serie_print["valor_total"] / 1e9).round(2)
serie_print.columns = ["N licitações", "Valor total (R$ bi)"]
print(serie_print.to_string())

> 📝 **Análise:**
>
> 2022 foi ano eleitoral no Brasil. Há sinais disso? Que meses concentram mais valor? Algum pico atípico que não casa com o padrão de fim de exercício? Lembre: a legislação eleitoral restringe contratações em alguns meses específicos.

## 7. Top fornecedores e concentração de mercado

Mercados saudáveis pulverizam contratos entre muitos fornecedores. **Concentração extrema é sinal de captura ou dependência problemática.** Usamos o índice HHI (Herfindahl-Hirschman) que é o padrão do CADE para análise de competição.

In [ ]:
# Fornecedores vêm da tabela ItemLicitacao (Nome Vencedor)
# Vamos agregar por fornecedor: quantos contratos, valor total
fornecedores = (
    df_itens.dropna(subset=["nome_vencedor", "valor_item"])
    .groupby("nome_vencedor")
    .agg(
        n_itens_vencidos=("valor_item", "count"),
        valor_total=("valor_item", "sum"),
    )
    .sort_values("valor_total", ascending=False)
)

print(f"Total de fornecedores únicos: {len(fornecedores):,}")
print(f"\nTop 15 por valor total recebido:\n")
top15 = fornecedores.head(15).copy()
top15["valor_total_mi"] = (top15["valor_total"]/1e6).round(2)
print(top15[["n_itens_vencidos", "valor_total_mi"]].rename(
    columns={"valor_total_mi": "Valor total (R$ mi)"}
).to_string())

In [ ]:
# Gráfico: Top 15 fornecedores por valor
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top15_plot = fornecedores.head(15)

# Por valor
axes[0].barh(range(len(top15_plot)), top15_plot["valor_total"]/1e6, color="darkred", alpha=0.8)
axes[0].set_yticks(range(len(top15_plot)))
axes[0].set_yticklabels([n[:40] + "..." if len(n) > 40 else n
                          for n in top15_plot.index], fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Valor total recebido (R$ milhões)")
axes[0].set_title("Top 15 fornecedores por VALOR total")

# Por quantidade de itens vencidos
top15_qtd = fornecedores.sort_values("n_itens_vencidos", ascending=False).head(15)
axes[1].barh(range(len(top15_qtd)), top15_qtd["n_itens_vencidos"], color="darkblue", alpha=0.8)
axes[1].set_yticks(range(len(top15_qtd)))
axes[1].set_yticklabels([n[:40] + "..." if len(n) > 40 else n
                          for n in top15_qtd.index], fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("Número de itens vencidos")
axes[1].set_title("Top 15 fornecedores por QUANTIDADE")

plt.tight_layout()
plt.show()

In [ ]:
# Curva de Pareto: % acumulado dos fornecedores vs % acumulado do valor
fornecedores_sorted = fornecedores.sort_values("valor_total", ascending=False)
total_valor = fornecedores_sorted["valor_total"].sum()
fornecedores_sorted["valor_acumulado_pct"] = (
    fornecedores_sorted["valor_total"].cumsum() / total_valor * 100
)
fornecedores_sorted["fornecedor_pct"] = (
    np.arange(1, len(fornecedores_sorted)+1) / len(fornecedores_sorted) * 100
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fornecedores_sorted["fornecedor_pct"],
        fornecedores_sorted["valor_acumulado_pct"],
        color="darkblue", linewidth=2)
ax.plot([0, 100], [0, 100], "k--", alpha=0.4, label="Distribuição uniforme (referência)")
ax.fill_between(fornecedores_sorted["fornecedor_pct"],
                fornecedores_sorted["valor_acumulado_pct"], alpha=0.2)
ax.set_xlabel("% acumulado de fornecedores (do maior para o menor)")
ax.set_ylabel("% acumulado do valor")
ax.set_title("Curva de Pareto: concentração de mercado")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Estatísticas-chave
top1_pct = fornecedores_sorted.iloc[:1]["valor_total"].sum() / total_valor * 100
top10_pct = fornecedores_sorted.iloc[:10]["valor_total"].sum() / total_valor * 100
top1pct_fornec_pct = (
    fornecedores_sorted.iloc[:int(len(fornecedores_sorted)*0.01)]["valor_total"].sum()
    / total_valor * 100
)
print(f"\nO maior fornecedor sozinho concentra: {top1_pct:.2f}% do valor")
print(f"Os 10 maiores concentram:               {top10_pct:.2f}% do valor")
print(f"Os 1% maiores fornecedores concentram:  {top1pct_fornec_pct:.2f}% do valor")

> 📝 **Análise:**
>
> A curva de Pareto está muito acima da linha diagonal (concentração alta) ou próxima dela (mercado pulverizado)? Quem são os maiores fornecedores e isso faz sentido (ex.: laboratórios farmacêuticos no Ministério da Saúde, construtoras em obras públicas)?

## 8. Análise de competição

Quantos participantes disputam cada licitação? **Licitação com 1 participante é um forte sinal de risco** — pode ser inexigibilidade legítima (fornecedor único), mas também pode ser direcionamento.

In [ ]:
# Conta participantes únicos por licitação (chave composta)
competicao = (
    df_par.groupby(CHAVE_LICITACAO)
    .agg(n_participantes=("codigo_participante", "nunique"))
    .reset_index()
)

print(f"Total de licitações com dados de participantes: {len(competicao):,}")
print(f"\nDistribuição de N participantes por licitação:")
print(competicao["n_participantes"].describe().round(2).to_string())

# Categorização
def cat_competicao(n):
    if n == 1: return "1 (sem competição)"
    elif n == 2: return "2"
    elif n == 3: return "3"
    elif n <= 5: return "4-5"
    elif n <= 10: return "6-10"
    else: return "11+"

competicao["categoria"] = competicao["n_participantes"].apply(cat_competicao)
print("\nLicitações por nível de competição:")
print(competicao["categoria"].value_counts().reindex(
    ["1 (sem competição)", "2", "3", "4-5", "6-10", "11+"]
).to_string())

In [ ]:
# Gráfico: distribuição de competição
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ordem_cat = ["1 (sem competição)", "2", "3", "4-5", "6-10", "11+"]
contagem = competicao["categoria"].value_counts().reindex(ordem_cat)

axes[0].bar(range(len(contagem)), contagem.values,
            color=["darkred", "orange", "gold", "yellowgreen", "green", "darkgreen"],
            edgecolor="black", linewidth=0.5)
axes[0].set_xticks(range(len(contagem)))
axes[0].set_xticklabels(ordem_cat, rotation=20, ha="right")
axes[0].set_title("Licitações por número de participantes")
axes[0].set_ylabel("Número de licitações")
for i, v in enumerate(contagem.values):
    if not pd.isna(v):
        axes[0].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=9)

# Histograma cru (cortado em 30 para legibilidade)
n_part_corte = competicao["n_participantes"].clip(upper=30)
axes[1].hist(n_part_corte, bins=30, color="teal", edgecolor="black", linewidth=0.3)
axes[1].set_title("Histograma de n_participantes (clip em 30)")
axes[1].set_xlabel("Número de participantes")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

> 📝 **Análise:**
>
> Que percentual das licitações tem 1 só participante? Isso é alto demais? Lembre que algumas modalidades (Inexigibilidade) admitem fornecedor único legalmente, então cruzar n_participantes com modalidade é o próximo passo natural.

## 9. Lista preliminar de candidatos a anomalia

**Esta é a entrega mais valiosa da EDA.** Identificar manualmente, com regras de negócio claras, licitações que parecem suspeitas. Vai servir como **referência semi-supervisionada**: depois compararemos com o output do Isolation Forest / LOF para validar os modelos.

**Critérios usados (combinados via OR):**
1. Valor acima do percentil 99 da modalidade
2. Concentração: licitação com 1 participante apenas
3. Acúmulo no limite de dispensa (entre R$ 17.500 e R$ 17.700)
4. Modalidade Dispensa/Inexigibilidade com valor > R$ 10 milhões (alto valor sem competição)

In [ ]:
# Critério 1: percentil 99 da modalidade
df_lic["valor_p99_modalidade"] = df_lic.groupby("modalidade_compra")["valor_licitacao"].transform(
    lambda s: s.quantile(0.99)
)
df_lic["criterio_1_acima_p99_modalidade"] = (
    df_lic["valor_licitacao"] > df_lic["valor_p99_modalidade"]
)

# Critério 2: 1 participante apenas (merge com competicao)
df_lic = df_lic.merge(
    competicao[CHAVE_LICITACAO + ["n_participantes"]],
    on=CHAVE_LICITACAO,
    how="left"
)
df_lic["criterio_2_sem_competicao"] = (df_lic["n_participantes"] == 1)

# Critério 3: fracionamento próximo ao teto de dispensa
df_lic["criterio_3_limite_dispensa"] = (
    (df_lic["valor_licitacao"] >= 17500) &
    (df_lic["valor_licitacao"] <= 17700)
)

# Critério 4: alto valor sem competição (modalidade)
modalidades_sem_comp = ["Dispensa de Licitação", "Inexigibilidade de Licitação"]
df_lic["criterio_4_alto_valor_sem_comp"] = (
    df_lic["modalidade_compra"].isin(modalidades_sem_comp) &
    (df_lic["valor_licitacao"] > 10_000_000)
)

# Flag final: pelo menos UM critério ativado
df_lic["candidato_anomalia"] = (
    df_lic["criterio_1_acima_p99_modalidade"] |
    df_lic["criterio_2_sem_competicao"] |
    df_lic["criterio_3_limite_dispensa"] |
    df_lic["criterio_4_alto_valor_sem_comp"]
)

# Contagem por critério
print("Quantidade de licitações por critério:")
print(f"  C1 (acima do P99 da modalidade):     {df_lic['criterio_1_acima_p99_modalidade'].sum():,}")
print(f"  C2 (sem competição, 1 participante): {df_lic['criterio_2_sem_competicao'].sum():,}")
print(f"  C3 (limite de dispensa):             {df_lic['criterio_3_limite_dispensa'].sum():,}")
print(f"  C4 (alto valor + sem competição):    {df_lic['criterio_4_alto_valor_sem_comp'].sum():,}")
print(f"\nTotal de candidatos (qualquer critério): {df_lic['candidato_anomalia'].sum():,} "
      f"({df_lic['candidato_anomalia'].sum()/len(df_lic)*100:.2f}%)")

In [ ]:
# Salva o dataset enriquecido com flags para uso na modelagem
caminho_saida = INTERIM_DIR / "licitacao_com_candidatos.parquet"
df_lic.to_parquet(caminho_saida, index=False)
print(f"💾 Dataset com flags de candidatos salvo em:\n  {caminho_saida}")
print(f"\nColunas adicionadas: candidato_anomalia + 4 critérios individuais")

In [ ]:
# Inspeção dos candidatos: amostra dos 20 candidatos com maior valor
print("Top 20 candidatos a anomalia por valor:")
print("=" * 80)

candidatos = df_lic[df_lic["candidato_anomalia"]].nlargest(20, "valor_licitacao")
for _, row in candidatos.iterrows():
    criterios_ativados = []
    if row["criterio_1_acima_p99_modalidade"]: criterios_ativados.append("C1")
    if row["criterio_2_sem_competicao"]: criterios_ativados.append("C2")
    if row["criterio_3_limite_dispensa"]: criterios_ativados.append("C3")
    if row["criterio_4_alto_valor_sem_comp"]: criterios_ativados.append("C4")
    
    print(f"\nR$ {row['valor_licitacao']:>18,.2f}  [{', '.join(criterios_ativados)}]")
    print(f"  Modalidade: {row['modalidade_compra']}")
    print(f"  Órgão:      {row['nome_orgao']}")
    print(f"  N particip: {row['n_participantes']}")
    print(f"  Objeto:     {str(row['objeto'])[:90]}...")

> 📝 **Análise final:**
>
> Comente os critérios escolhidos e sua justificativa metodológica. Olhe a amostra dos 20 candidatos: eles parecem genuinamente suspeitos ou tem muito falso-positivo? Lembre-se do caso Eculizumabe — alto valor + sem competição pode ser perfeitamente legal. **Esta seção é a base da seção de Metodologia do relatório final.**

---

## Próximos passos

Com os achados desta EDA, próximas etapas:

1. **Engenharia de features** (`03_features.ipynb`) — construir as variáveis finais para modelagem agregando das 4 tabelas
2. **Modelagem não-supervisionada** (`04_modelagem_anomalia.ipynb`) — Isolation Forest e LOF
3. **Comparação com candidatos** — validar se os modelos "concordam" com nossa lista preliminar
4. **Modelagem supervisionada** (`05_modelagem_supervisionada.ipynb`) — usando `candidato_anomalia` como label proxy